In [1]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

tm = pd.read_sql("SELECT * FROM hpce.markov_transition_matrix", engine)

states = sorted(set(tm["prev_risk_band"]) | set(tm["risk_band"]))
state_to_idx = {s:i for i,s in enumerate(states)}

P = np.zeros((len(states), len(states)))
for _, r in tm.iterrows():
    i = state_to_idx[r["prev_risk_band"]]
    j = state_to_idx[r["risk_band"]]
    P[i, j] = float(r["p"])

P_df = pd.DataFrame(P, index=states, columns=states)
print("1-step Transition Matrix P:")
display(P_df)

def k_step(P, k):
    return np.linalg.matrix_power(P, k)

for k in [7, 30]:
    Pk = k_step(P, k)
    Pk_df = pd.DataFrame(Pk, index=states, columns=states)
    print(f"\n{k}-step transition probabilities:")
    display(Pk_df)

# Steady state (stationary distribution)
# Solve pi = pi P  with sum(pi)=1 using eigenvector of transpose with eigenvalue 1
eigvals, eigvecs = np.linalg.eig(P.T)
idx = np.argmin(np.abs(eigvals - 1))
pi = np.real(eigvecs[:, idx])
pi = pi / pi.sum()
pi = np.maximum(pi, 0)
pi = pi / pi.sum()

steady = pd.Series(pi, index=states, name="steady_state_prob")
print("\nSteady-state distribution:")
display(steady.sort_values(ascending=False))

1-step Transition Matrix P:


,High Risk,Low Risk,Medium Risk
High Risk,0.298246,0.473684,0.228070
Low Risk,0.013938,0.963479,0.022583
Medium Risk,0.016715,0.043228,0.940058



7-step transition probabilities:


,High Risk,Low Risk,Medium Risk
High Risk,0.020861,0.658263,0.320876
Low Risk,0.019934,0.837451,0.142615
Medium Risk,0.022343,0.284839,0.692818



30-step transition probabilities:


,High Risk,Low Risk,Medium Risk
High Risk,0.020734,0.654598,0.324668
Low Risk,0.020624,0.679864,0.299512
Medium Risk,0.020964,0.601903,0.377134



Steady-state distribution:


Low Risk       0.653980
Medium Risk    0.325283
High Risk      0.020737
Name: steady_state_prob, dtype: float64

In [2]:
P_scn = P.copy()

if "High Risk" in state_to_idx and "Low Risk" in state_to_idx:
    h = state_to_idx["High Risk"]
    l = state_to_idx["Low Risk"]

    boost = 0.05
    P_scn[h, l] = min(1.0, P_scn[h, l] + boost)

    row_sum = P_scn[h, :].sum()
    P_scn[h, :] = P_scn[h, :] / row_sum

def steady_state(Pm):
    eigvals, eigvecs = np.linalg.eig(Pm.T)
    idx = np.argmin(np.abs(eigvals - 1))
    pi = np.real(eigvecs[:, idx])
    pi = pi / pi.sum()
    pi = np.maximum(pi, 0)
    return pi / pi.sum()

steady_base = pd.Series(steady_state(P), index=states, name="base")
steady_scn  = pd.Series(steady_state(P_scn), index=states, name="scenario")

compare = pd.concat([steady_base, steady_scn], axis=1)
compare["delta"] = compare["scenario"] - compare["base"]
display(compare.sort_values("delta", ascending=False))

,base,scenario,delta
Low Risk,0.653980,0.658112,0.004132
High Risk,0.020737,0.020319,-0.000418
Medium Risk,0.325283,0.321569,-0.003714


In [3]:
row_sums = P.sum(axis=1)
print("Row sums:", dict(zip(states, row_sums)))

Row sums: {'High Risk': np.float64(1.0), 'Low Risk': np.float64(1.0), 'Medium Risk': np.float64(1.0)}
